# 邮件

本 Notebook 展示了如何加载电子邮件 (`.eml`) 或 `Microsoft Outlook` (`.msg`) 文件。

请参阅[本指南](/docs/integrations/providers/unstructured/)，了解有关在本地设置 Unstructured 的更多说明，包括设置所需的系统依赖项。

## 使用非结构化数据

Unstructured is a Python library that helps you process unstructured data like PDFs, HTML, DOCX, EML, TXT, and more. It can extract text, tables, and other information from these documents, making it easier to work with them in downstream NLP tasks.

Unstructured 是一个 Python 库，可帮助您处理非结构化数据，如 PDF、HTML、DOCX、EML、TXT 等。它可以从这些文档中提取文本、表格和其他信息，从而更轻松地在下游 NLP 任务中使用它们。

Here's a quick overview of how to use Unstructured:

以下是如何使用 Unstructured 的快速概述：

### Installation

### 安装

First, install the library:

首先，安装该库：

```bash
pip install unstructured
```

### Basic Usage

### 基本用法

Let's say you have a PDF file named `example.pdf`. You can use Unstructured to extract text from it like this:

假设您有一个名为 `example.pdf` 的 PDF 文件。您可以使用 Unstructured 像这样从中提取文本：

```python
from unstructured.partition.auto import partition

# Partition the PDF file
elements = partition(filename="example.pdf")

# Print the extracted text
for element in elements:
    print(element)
```

The `partition` function automatically detects the file type and uses the appropriate parser to extract the content. The result is a list of `Element` objects, each representing a piece of content (like a paragraph, title, or table) from the document.

`partition` 函数会自动检测文件类型并使用适当的解析器提取内容。结果是一个 `Element` 对象列表，每个对象代表文档中的一块内容（如段落、标题或表格）。

### Handling Different File Types

### 处理不同文件类型

Unstructured supports a wide range of file types. You can simply pass the filename to the `partition` function, and it will handle the rest.

Unstructured 支持多种文件类型。您只需将文件名传递给 `partition` 函数，它就会处理其余部分。

```python
# For an HTML file
elements_html = partition(filename="example.html")

# For a DOCX file
elements_docx = partition(filename="example.docx")

# For an EML file
elements_eml = partition(filename="example.eml")
```

### Extracting Tables

### 提取表格

Unstructured has specific capabilities for extracting tables from documents, especially PDFs.

Unstructured 在从文档（尤其是 PDF）中提取表格方面具有特定功能。

```python
from unstructured.partition.pdf import partition_pdf

# Partition a PDF and specify that you want to extract tables
elements = partition_pdf(filename="example.pdf", infer_table_structure=True)

for element in elements:
    if element.category == "Table":
        print("Found a table:")
        # The table content is often in a pandas DataFrame
        print(element.metadata.text_as_html)
```

The `infer_table_structure=True` argument tells Unstructured to try and identify table structures within the PDF. Table data is often returned as an HTML string or can be processed into other formats.

`infer_table_structure=True` 参数告诉 Unstructured 尝试识别 PDF 中的表格结构。表格数据通常以 HTML 字符串形式返回，或者可以处理成其他格式。

### Using with LangChain

### 与 LangChain 一起使用

Unstructured integrates seamlessly with LangChain, a popular framework for developing applications powered by language models. You can use Unstructured's `partition` function as a document loader for LangChain.

Unstructured 与 LangChain（一个用于开发由语言模型驱动的应用程序的流行框架）无缝集成。您可以将 Unstructured 的 `partition` 函数用作 LangChain 的文档加载器。

```python
from langchain_community.document_loaders import UnstructuredFileLoader

# Load a document using UnstructuredFileLoader
loader = UnstructuredFileLoader("example.pdf")
docs = loader.load()

# Now 'docs' is a list of LangChain Document objects
for doc in docs:
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Content: {doc.page_content[:100]}...") # Print first 100 characters
```

This allows you to easily ingest unstructured data into your LangChain pipelines for tasks like question answering, summarization, and more.

这使您可以轻松地将非结构化数据摄取到您的 LangChain 管道中，用于问答、摘要等任务。

### Advanced Options

### 高级选项

The `partition` function offers various options for customization:

`partition` 函数提供了各种自定义选项：

*   **`strategy`**: Controls the partitioning strategy (e.g., `fast`, `hi_res`). `hi_res` uses more advanced models for better accuracy but is slower.
*   **`languages`**: Specify the language(s) of the document for better OCR and text extraction.
*   **`chunking_strategy`**: Define how the document should be split into smaller chunks.

*   **`strategy`**：控制分区策略（例如，`fast`、`hi_res`）。`hi_res` 使用更高级的模型以获得更好的准确性，但速度较慢。
*   **`languages`**：指定文档的语言，以获得更好的 OCR 和文本提取效果。
*   **`chunking_strategy`**：定义文档应如何拆分成更小的块。

```python
from unstructured.partition.auto import partition

elements = partition(
    filename="example.pdf",
    strategy="hi_res",
    languages=["eng"], # Specify English
    chunking_strategy="by_title"
)

for element in elements:
    print(element)
```

Refer to the [Unstructured documentation](https://unstructured-io.github.io/unstructured/main/inference.html) for a complete list of options and more detailed examples.

请参阅 [Unstructured 文档](https://unstructured-io.github.io/unstructured/main/inference.html) 以获取选项的完整列表和更详细的示例。

In [ ]:
%pip install --upgrade --quiet unstructured

In [3]:
from langchain_community.document_loaders import UnstructuredEmailLoader

loader = UnstructuredEmailLoader("./example_data/fake-email.eml")

data = loader.load()

data

[Document(page_content='This is a test email to use for unit tests.\n\nImportant points:\n\nRoses are red\n\nViolets are blue', metadata={'source': './example_data/fake-email.eml'})]

### 保留元素

在底层，Unstructured 会为不同的文本块创建不同的“元素”。默认情况下，我们会将它们合并在一起，但您可以通过指定 `mode="elements"` 来轻松保留这种分离。

In [4]:
loader = UnstructuredEmailLoader("example_data/fake-email.eml", mode="elements")

data = loader.load()

data[0]

Document(page_content='This is a test email to use for unit tests.', metadata={'source': 'example_data/fake-email.eml', 'file_directory': 'example_data', 'filename': 'fake-email.eml', 'last_modified': '2022-12-16T17:04:16-05:00', 'sent_from': ['Matthew Robinson <mrobinson@unstructured.io>'], 'sent_to': ['Matthew Robinson <mrobinson@unstructured.io>'], 'subject': 'Test Email', 'languages': ['eng'], 'filetype': 'message/rfc822', 'category': 'NarrativeText'})

### 处理附件

您可以通过在构造函数中设置 `process_attachments=True` 来使用 `UnstructuredEmailLoader` 处理附件。默认情况下，附件将使用 `unstructured` 的 `partition` 函数进行分区。您可以通过将函数传递给 `attachment_partitioner` 关键字参数来使用不同的分区函数。

In [5]:
loader = UnstructuredEmailLoader(
    "example_data/fake-email.eml",
    mode="elements",
    process_attachments=True,
)

data = loader.load()

data[0]

Document(page_content='This is a test email to use for unit tests.', metadata={'source': 'example_data/fake-email.eml', 'file_directory': 'example_data', 'filename': 'fake-email.eml', 'last_modified': '2022-12-16T17:04:16-05:00', 'sent_from': ['Matthew Robinson <mrobinson@unstructured.io>'], 'sent_to': ['Matthew Robinson <mrobinson@unstructured.io>'], 'subject': 'Test Email', 'languages': ['eng'], 'filetype': 'message/rfc822', 'category': 'NarrativeText'})

## 使用 OutlookMessageLoader

The `OutlookMessageLoader` is a specialized loader designed to parse `.msg` files, which are the standard format for saving individual emails in Microsoft Outlook. This loader allows you to extract the content and metadata from these email files.

### Installation

To use the `OutlookMessageLoader`, you first need to install the `llamaindex-addons-contrib` package.

```bash
pip install llama-index-addons-contrib
```

### Usage

Here's a basic example of how to use the `OutlookMessageLoader` to load an email from a `.msg` file:

```python
from llama_index.readers.outlook import OutlookMessageLoader

# Initialize the loader
loader = OutlookMessageLoader()

# Load the email message from a .msg file
# Replace 'path/to/your/email.msg' with the actual path to your .msg file
documents = loader.load_data("path/to/your/email.msg")

# Print the content of the loaded document
for doc in documents:
    print(doc.text)
```

### Options

The `OutlookMessageLoader` can be initialized with several options to customize its behavior:

*   **`include_attachments`** (bool, optional): If `True`, the loader will attempt to extract and include the content of email attachments. Defaults to `False`.
*   **`attachment_path`** (str, optional): If `include_attachments` is `True`, this path specifies where to save the extracted attachment files. If not provided, attachments will be saved in a temporary directory.
*   **`recursive`** (bool, optional): If `True`, the loader will recursively load `.msg` files from subdirectories within the specified path. Defaults to `False`.

Here's an example demonstrating the use of these options:

```python
from llama_index.readers.outlook import OutlookMessageLoader

# Initialize the loader to include attachments and save them to a specific directory
loader = OutlookMessageLoader(
    include_attachments=True,
    attachment_path="./attachments"
)

# Load email messages from a directory, recursively
# Replace 'path/to/your/emails/' with the actual path to your email directory
documents = loader.load_data("path/to/your/emails/", recursive=True)

# Process the loaded documents
for doc in documents:
    print(f"Subject: {doc.metadata.get('subject')}")
    print(f"From: {doc.metadata.get('from')}")
    print(f"Content:\n{doc.text}\n---")
```

This example shows how to load multiple `.msg` files from a directory, including their attachments, and then access metadata such as the subject and sender.

In [ ]:
%pip install --upgrade --quiet extract_msg

In [7]:
from langchain_community.document_loaders import OutlookMessageLoader

loader = OutlookMessageLoader("example_data/fake-email.msg")

data = loader.load()

data[0]

Document(page_content='This is a test email to experiment with the MS Outlook MSG Extractor\r\n\r\n\r\n-- \r\n\r\n\r\nKind regards\r\n\r\n\r\n\r\n\r\nBrian Zhou\r\n\r\n', metadata={'source': 'example_data/fake-email.msg', 'subject': 'Test for TIF files', 'sender': 'Brian Zhou <brizhou@gmail.com>', 'date': datetime.datetime(2013, 11, 18, 0, 26, 24, tzinfo=zoneinfo.ZoneInfo(key='America/Los_Angeles'))})